In [15]:
!pip install groq

In [16]:
from groq import Groq
from google.colab import userdata
import json
import csv

client_groq = Groq(api_key=userdata.get('GROQ_API_KEY'))
client = Groq(api_key=userdata.get('GROQ_API_KEY'))


In [17]:
def analyse_customer_structured(client, customer):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a banking risk analyst.
                Always respond with ONLY valid JSON, no other text.
                Use exactly this structure:
                {
                    "customer_name": "string",
                    "risk_score": number between 1-10,
                    "risk_level": "Low" or "Medium" or "High",
                    "reason": "one sentence explanation",
                    "recommendation": "one sentence action"
                }"""
            },
            {
                "role": "user",
                "content": f"""Analyse this customer:
                Name: {customer['name']}
                Balance: € {customer['balance']}
                Transactions: {customer['transactions']}
                Active: {customer['is_active']}"""
            }
        ]
    )
    return json.loads(response.choices[0].message.content)




In [18]:
from groq import Groq
import json
from google.colab import userdata



def generate_user_story(requirement):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a senior AI Product Owner with
                10 years experience in banking and financial services.

                When given a business requirement, respond with ONLY
                valid JSON in exactly this structure:
                {
                    "user_story": "As a [role] I want [feature] so that [benefit]",
                    "acceptance_criteria": [
                        "Given [context] When [action] Then [outcome]",
                        "Given [context] When [action] Then [outcome]",
                        "Given [context] When [action] Then [outcome]"
                    ],
                    "definition_of_done": [
                        "criterion 1",
                        "criterion 2",
                        "criterion 3"
                    ],
                    "story_points": number,
                    "priority": "High" or "Medium" or "Low"
                }"""
            },
            {
                "role": "user",
                "content": f"Generate a user story for this requirement: {requirement}"
            }
        ]
    )
    return json.loads(response.choices[0].message.content)



In [19]:
from groq import Groq
from google.colab import userdata
import json


def check_eu_ai_act(use_case):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are an EU AI Act compliance expert
                specialising in financial services.

                Analyse AI use cases and classify them under the EU AI Act.

                Respond with ONLY valid JSON in exactly this structure:
                {
                    "use_case_summary": "brief summary of the use case",
                    "risk_tier": "Unacceptable" or "High Risk" or "Limited Risk" or "Minimal Risk",
                    "risk_tier_reason": "why this tier applies",
                    "compliance_requirements": [
                        "requirement 1",
                        "requirement 2",
                        "requirement 3"
                    ],
                    "red_flags": [
                        "red flag 1",
                        "red flag 2"
                    ],
                    "recommended_actions": [
                        "action 1",
                        "action 2",
                        "action 3"
                    ],
                    "compliant_by": "date by which compliance is required"
                }"""
            },
            {
                "role": "user",
                "content": f"Analyse this AI use case for EU AI Act compliance: {use_case}"
            }
        ]
    )
    return json.loads(response.choices[0].message.content)



In [20]:
def assess_taxonomy(client):

    if not client['nace_code']:
        return {
            "client_id": client['client_id'],
            "client_name": client['client_name'],
            "assessment_tier": "CANNOT ASSESS",
            "taxonomy_eligible": None,
            "taxonomy_aligned": None,
            "substantial_contribution": None,
            "dnsh_assessment": None,
            "minimum_safeguards": None,
            "revenue_kpi": None,
            "capex_kpi": None,
            "gaps": ["Insufficient data for any assessment"],
            "recommendations": ["Collect NACE code, sector, GHG data and safeguards confirmation from client"]
        }

    # Build context
    capex_display = f"€ {client['capex_eur']:,}" if client['capex_eur'] else 'Not provided'

    context = f"""
    Client: {client['client_name']}
    Sector: {client['sector'] or 'Unknown'}
    NACE Code: {client['nace_code']}
    Main Purpose: {client['main_purpose'] or 'Not provided'}
    Sub Purpose: {client['sub_purpose'] or 'Not provided'}
    Product Type: {client['product_type']}
    Reporting Year: {client['reporting_year']}
    Substantial Contribution Objective: {client['substantial_contribution_objective'] or 'Not provided'}
    Revenue Eligible %: {client['revenue_eligible_pct'] or 'Not provided'}
    CapEx Eligible %: {client['capex_eligible_pct'] or 'Not provided'}
    GHG Scope 1+2 (tonnes): {client['ghg_scope1_2_tonnes'] or 'Not provided'}
    Minimum Safeguards Compliant: {client['minimum_safeguards_compliant']}
    Annual Turnover: € {client['annual_turnover_eur']:,}
    CapEx: {capex_display}
    """

    if client['nace_code'] and client['main_purpose'] and client['sub_purpose'] and client['ghg_scope1_2_tonnes']:
        instruction = "Perform a complete 4-step EU Taxonomy assessment."
    elif client['nace_code'] and client['main_purpose']:
        instruction = "Perform a partial EU Taxonomy assessment. Check eligibility and substantial contribution only."
    else:
        instruction = "Perform an eligibility check only based on NACE code."

    response = client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a senior EU Taxonomy compliance expert at a Dutch bank.
You follow the official EU Taxonomy Compass 4-step framework strictly.

CRITICAL RULES:
- Diesel or fossil fuel transport is NOT taxonomy aligned
- Traditional blast furnace steel is NOT taxonomy aligned
- Only activities meeting Technical Screening Criteria are aligned
- Be conservative — when in doubt mark as not aligned

Respond with ONLY valid JSON. No preamble, no markdown.

                Use exactly this structure:
                {
                    "assessment_tier": "FULL" or "PARTIAL" or "ELIGIBILITY",
                    "taxonomy_eligible": true or false,
                    "taxonomy_aligned": true or false or null,
                    "substantial_contribution": {
                        "objective": "objective name",
                        "meets_criteria": true or false,
                        "reasoning": "one sentence"
                    },
                    "dnsh_assessment": {
                        "climate_mitigation": "Pass" or "Fail" or "Incomplete",
                        "climate_adaptation": "Pass" or "Fail" or "Incomplete",
                        "water": "Pass" or "Fail" or "Incomplete",
                        "circular_economy": "Pass" or "Fail" or "Incomplete",
                        "pollution": "Pass" or "Fail" or "Incomplete",
                        "biodiversity": "Pass" or "Fail" or "Incomplete",
                        "overall": "Pass" or "Fail" or "Incomplete"
                    },
                    "minimum_safeguards": "Pass" or "Fail" or "Incomplete",
                    "revenue_kpi": number or null,
                    "capex_kpi": number or null,
                    "gaps": ["gap 1", "gap 2"],
                    "recommendations": ["action 1", "action 2"]
                }"""
            },
            {
                "role": "user",
                "content": f"{instruction}\n\nClient data:\n{context}"
            }
        ]
    )

    # Debug — see raw response
    raw = response.choices[0].message.content
    print(f"Raw response: {raw[:200]}")

    # Clean and parse
    try:
        clean = raw.strip()
        if clean.startswith("```"):
            clean = clean.split("```")[1]
            if clean.startswith("json"):
                clean = clean[4:]
        result = json.loads(clean.strip())
        result['client_id'] = client['client_id']
        result['client_name'] = client['client_name']
        return result
    except Exception as e:
        print(f"Parse error: {e}")
        print(f"Full raw response: {raw}")
        return None


In [ ]:
from google.colab import files

def show_menu():
    print("\n" + "=" * 40)
    print("🏦 AI BANKING PLATFORM")
    print("=" * 40)
    print("1. Customer Risk Assessment")
    print("2. Generate User Stories")
    print("3. EU AI Act Checker")
    print("4. EU Taxonomy Assessment")
    print("5. Document Q&A (RAG)")
    print("0. Exit")
    print("=" * 40)
    choice = input("Choose a tool (0-5): ")
    return choice

def run_platform():
    print("\n👋 Welcome to the AI Banking Platform")

    while True:
        choice = show_menu()

        if choice == "1":
            print("\n🏦 CUSTOMER RISK ASSESSMENT")
            print("-" * 40)
            try:
                name = input("Customer name: ")
                balance = float(input("Balance (€): "))
                transactions = int(input("Transactions: "))
                active = input("Active? (yes/no): ").lower() == "yes"
                customer = {
                    "name": name,
                    "balance": balance,
                    "transactions": transactions,
                    "is_active": active
                }
                print("\n⏳ Analysing...")
                result = analyse_customer_structured(client_groq, customer)
                print(f"\n📋 {name.upper()}")
                print(f"Risk Score:     {result['risk_score']}/10")
                print(f"Risk Level:     {result['risk_level']}")
                print(f"Reason:         {result['reason']}")
                print(f"Recommendation: {result['recommendation']}")
            except ValueError:
                print("\n❌ Invalid input — balance and transactions must be numbers")
            except Exception as e:
                print(f"\n❌ Something went wrong: {e}")

        elif choice == "2":
            print("\n📋 USER STORY GENERATOR")
            print("-" * 40)
            try:
                requirement = input("Enter business requirement: ")
                print("\n⏳ Generating...")
                story = generate_user_story(requirement)
                print(f"\n✅ USER STORY:")
                print(f"{story['user_story']}")
                print(f"\nPriority:     {story['priority']}")
                print(f"Story Points: {story['story_points']}")
                print(f"\nACCEPTANCE CRITERIA:")
                for ac in story['acceptance_criteria']:
                    print(f"  → {ac}")
                print(f"\nDEFINITION OF DONE:")
                for dod in story['definition_of_done']:
                    print(f"  ☐ {dod}")
            except Exception as e:
                print(f"\n❌ Something went wrong: {e}")

        elif choice == "3":
            print("\n⚖️  EU AI ACT CHECKER")
            print("-" * 40)
            try:
                use_case = input("Describe your AI use case: ")
                print("\n⏳ Assessing...")
                result = check_eu_ai_act(use_case)
                print(f"\n{result['risk_tier']}")
                print(f"Reason: {result['risk_tier_reason']}")
                print(f"\nCOMPLIANCE REQUIREMENTS:")
                for req in result['compliance_requirements']:
                    print(f"  → {req}")
                print(f"\nRED FLAGS:")
                for flag in result['red_flags']:
                    print(f"  ⚠️  {flag}")
                print(f"\nRECOMMENDED ACTIONS:")
                for action in result['recommended_actions']:
                    print(f"  → {action}")
            except Exception as e:
                print(f"\n❌ Something went wrong: {e}")

        elif choice == "4":
            print("\n🌱 EU TAXONOMY ASSESSMENT")
            print("-" * 40)
            try:
                client_id = input("Client ID: ")
                client_name = input("Client name: ")
                sector = input("Sector: ")
                nace_code = input("NACE code: ")
                main_purpose = input("Main purpose: ")
                sub_purpose = input("Sub purpose (or Enter to skip): ")
                loan_amount = float(input("Loan amount (€): "))
                ghg = input("GHG Scope 1+2 tonnes (or Enter to skip): ")

                taxonomy_client = {
                    "client_id": client_id,
                    "client_name": client_name,
                    "sector": sector,
                    "nace_code": nace_code,
                    "main_purpose": main_purpose,
                    "sub_purpose": sub_purpose or None,
                    "product_type": "Term Loan",
                    "country": "Netherlands",
                    "annual_turnover_eur": 0,
                    "capex_eur": 0,
                    "loan_amount_eur": loan_amount,
                    "reporting_year": 2024,
                    "substantial_contribution_objective": "Climate Change Mitigation",
                    "revenue_eligible_pct": None,
                    "capex_eligible_pct": None,
                    "ghg_scope1_2_tonnes": float(ghg) if ghg else None,
                    "minimum_safeguards_compliant": True
                }

                print("\n⏳ Assessing...")
                result = assess_taxonomy(taxonomy_client)
                print(f"\n🌱 {client_name.upper()}")
                print(f"Eligible:   {result['taxonomy_eligible']}")
                print(f"Aligned:    {result['taxonomy_aligned']}")
                print(f"Tier:       {result['assessment_tier']}")
                if result.get('dnsh_assessment'):
                    print(f"DNSH:       {result['dnsh_assessment']['overall']}")
                print(f"Safeguards: {result['minimum_safeguards']}")
                if result.get('gaps'):
                    print(f"\nGAPS:")
                    for gap in result['gaps']:
                        print(f"  ⚠️  {gap}")
                if result.get('recommendations'):
                    print(f"\nRECOMMENDATIONS:")
                    for rec in result['recommendations']:
                        print(f"  → {rec}")
            except ValueError:
                print("\n❌ Invalid input — loan amount must be a number")
            except Exception as e:
                print(f"\n❌ Something went wrong: {e}")

        elif choice == "5":
            print("\n📄 DOCUMENT Q&A (RAG)")
            print("-" * 40)
            try:
                print("⏳ Please upload a PDF document...")
                uploaded = files.upload()
                filename = list(uploaded.keys())[0]

                print(f"\n⏳ Processing {filename}...")
                doc_text = read_pdf(filename)
                doc_chunks = split_into_chunks(doc_text)
                doc_embeddings = embed_chunks(doc_chunks)

                print(f"✅ Document ready!")
                print(f"📄 {len(doc_text)} characters · {len(doc_chunks)} chunks")
                print(f"\nAsk questions about your document.")
                print(f"Type 'exit' to go back to menu.\n")

                while True:
                    question = input("❓ Your question: ")
                    if question.lower() == 'exit':
                        break
                    print("\n⏳ Searching document...")
                    answer = ask_document(question, doc_chunks, doc_embeddings)
                    print(f"\n💬 {answer}\n")
                    print("-" * 40)

            except Exception as e:
                print(f"\n❌ Something went wrong: {e}")

        elif choice == "0":
            print("\n👋 Goodbye!")
            break
        else:
            print("\n❌ Invalid choice. Try again.")

# Run the platform
run_platform()


👋 Welcome to the AI Banking Platform

🏦 AI BANKING PLATFORM
1. Customer Risk Assessment
2. Generate User Stories
3. EU AI Act Checker
4. EU Taxonomy Assessment
5. Document Q&A (RAG)
0. Exit
Choose a tool (0-5): 5

📄 DOCUMENT Q&A (RAG)
----------------------------------------
⏳ Please upload a PDF document...


Saving esg_policy.pdf to esg_policy (1).pdf

⏳ Processing esg_policy (1).pdf...
✅ Document ready!
📄 1062 characters · 8 chunks

Ask questions about your document.
Type 'exit' to go back to menu.

❓ Your question: what is reporting period

⏳ Searching document...

💬 The reporting period is annually.

----------------------------------------
❓ Your question: exit

🏦 AI BANKING PLATFORM
1. Customer Risk Assessment
2. Generate User Stories
3. EU AI Act Checker
4. EU Taxonomy Assessment
5. Document Q&A (RAG)
0. Exit


In [22]:
!pip install pypdf sentence-transformers numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.0 MB/s eta 0:00:00


In [23]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
print("⏳ Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ RAG components ready")

⏳ Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ RAG components ready


In [24]:
def read_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text()
    return full_text

def split_into_chunks(text, chunk_size=200, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

def embed_chunks(chunks):
    return embedding_model.encode(chunks)

def find_best_chunk(question, chunks, chunk_embeddings):
    question_embedding = embedding_model.encode([question])
    similarities = []
    for i, chunk_emb in enumerate(chunk_embeddings):
        score = np.dot(question_embedding[0], chunk_emb)
        similarities.append((score, chunks[i]))
    similarities.sort(key=lambda x: x[0], reverse=True)
    return similarities[0][1]

def ask_document(question, chunks, chunk_embeddings):
    relevant_chunk = find_best_chunk(question, chunks, chunk_embeddings)

    response = client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a document analysis expert.
                Answer questions based ONLY on the provided document excerpt.
                If the answer is not in the document say so clearly.
                Be concise and precise."""
            },
            {
                "role": "user",
                "content": f"""Document excerpt:
                {relevant_chunk}

                Question: {question}"""
            }
        ]
    )
    return response.choices[0].message.content

print("✅ RAG functions ready")

✅ RAG functions ready


In [25]:
from google.colab import files

# Upload a PDF
print("📄 Please upload a PDF document...")
uploaded = files.upload()

# Get the filename
filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {filename}")

# Process the document
print("⏳ Processing document...")
text = read_pdf(filename)
chunks = split_into_chunks(text)
chunk_embeddings = embed_chunks(chunks)

print(f"✅ Document processed!")
print(f"📄 Total text: {len(text)} characters")
print(f"📦 Total chunks: {len(chunks)}")

📄 Please upload a PDF document...


Saving esg_policy.pdf to esg_policy.pdf
✅ Uploaded: esg_policy.pdf
⏳ Processing document...
✅ Document processed!
📄 Total text: 1062 characters
📦 Total chunks: 8


In [26]:
# Test questions
questions = [
    "What is the deadline for ESG reporting?",
    "Which sectors require DNSH assessment?",
    "What metrics must clients report?",
    "What happens if a client doesn't submit their report?"
]

print("🤖 DOCUMENT Q&A — Northern Europe Bank ESG Policy")
print("=" * 55)

for question in questions:
    print(f"\n❓ {question}")
    answer = ask_document(question, chunks, chunk_embeddings)
    print(f"💬 {answer}")
    print("-" * 55)

🤖 DOCUMENT Q&A — Northern Europe Bank ESG Policy

❓ What is the deadline for ESG reporting?
💬 The deadline for ESG reporting is 31 March each year.
-------------------------------------------------------

❓ Which sectors require DNSH assessment?
💬 High Risk sectors, specifically Energy, Manufacturing, and Transport, require DNSH assessment.
-------------------------------------------------------

❓ What metrics must clients report?
💬 According to the document, clients must report the following metrics: 
- GHG Scope 1 and Scope 2 emissions in tonnes CO2 equivalent.
-------------------------------------------------------

❓ What happens if a client doesn't submit their report?
💬 According to the document, if a client doesn't submit their report, they will be flagged for relationship review.
-------------------------------------------------------
